# Glutamate response analysis run

Use this notebook to load one session, run activation/tuning/sequence analyses, and save standardized output tables.

In [1]:
import os
import glob
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from vip_slap2_analysis.glutamate.analysis import (
    GlutamateAnalysisConfig,
    resolve_glutamate_analysis_paths,
    run_glutamate_analysis,
)

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
target_mice = [
    803496,804730,804733,810196,
    809047,803121,
    826033,838410,834788
]

In [4]:
registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

In [5]:
assets[0]

SessionAssets(session_id='803496_2025-07-25_13-02-10', subject_id=803496, session_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496'), summary_mat=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10_slap2_2026-01-18_05-53-08/source_extraction/ExperimentSummary/SummaryLoCo-260117-185357.mat'), bonsai_event_log_csv=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10/behavior/VCO1_Behavior.harp/bonsai_event_log.csv'), harp_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10/behavior/VCO1_Behavior.harp'), photodiode_pkl=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10/behavior/VCO1_Behavior.harp/extracted_files

In [6]:
for asset in assets[:]:
    session_root = asset.session_dir
    paths = resolve_glutamate_analysis_paths(session_root)
    print(paths)

    config = GlutamateAnalysisConfig(
        tuning_method="fve",                 # or "manova" or "hybrid"
        manova_stat="Wilks' lambda",
        manova_max_timepoints=10,
        manova_use_post_only=True,
        random_seed=0,

        # --- new sequence-analysis settings ---
        sequence_peak_window_samples=10,     # rolling peak window inside post-image epoch
        sequence_n_quantile_bins=3,          # early / mid / late
        sequence_min_count_per_position=1,   # keep sparse late positions, but bin them
        sequence_label_slope_frac=0.15,      # relative threshold for calling non-stable motifs
        sequence_label_min_abs_slope=5,   # absolute slope floor for labels
    )

    results = run_glutamate_analysis(session_root, config=config)

    {k: v.shape for k, v in results.items() if hasattr(v, "shape")}

    activation_summary = results["activation_summary_table"]
    tuning_per_image = results["tuning_per_image_table"]
    tuning_summary = results["tuning_summary_table"]
    sequence_position = results["sequence_position_table"]
    sequence_per_image = results["sequence_per_image_table"]
    sequence_summary = results["sequence_summary_table"]

    display(activation_summary.head())
    display(tuning_summary.head())
    display(sequence_per_image.head())
    display(sequence_position.head())
    display(sequence_summary.head())

    output_dir = resolve_glutamate_analysis_paths(session_root).output_dir
    print(output_dir)
    print(sorted(p.name for p in output_dir.iterdir()))

GlutamateAnalysisPaths(single_trial_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_single_trial_df.npz'), mean_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_mean_df.npz'), sequence_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_sequence_df.npz'), output_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_analysis'))


,session_id,subject_id,dmd,synapse_id,stimulus_family,n_events,median_delta_auc,mean_delta_auc,effect_direction,p_value,response_class,q_value_within_synapse
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0000,image,2277,-20.015213,-23.085767,none,3.356907e-02,no_change,1.007072e-01
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0001,image,2277,11.062568,10.113008,none,1.885622e-01,no_change,3.431282e-01
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,image,2277,62.247024,75.345806,up,1.833899e-10,activated,5.501698e-10
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0003,image,2277,51.070708,94.594795,up,5.779422e-15,activated,1.733827e-14
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0004,image,2277,-94.736861,-159.029189,down,1.601894e-34,deactivated,4.805682e-34


,session_id,subject_id,dmd,synapse_id,n_image_trials,n_images_tested,fve_image,p_shuffle_fve,p_kw,p_manova,...,preferred_image,preferred_mean,preferred_median,preferred_vs_rest_effect,preferred_vs_next_effect,is_tuned,tuning_method,q_shuffle_fve,q_kw,q_manova
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,2277,7,0.016428,0.000500,6.161776e-06,1.607803e-06,...,imk01643,209.877494,194.113103,148.201225,79.053404,False,fve,0.000778,1.247760e-05,4.069752e-06
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0003,2277,7,0.029130,0.000500,2.518969e-11,1.489699e-14,...,imk01378,186.991674,110.479186,69.505918,6.580640,False,fve,0.000778,7.847557e-11,7.541602e-14
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0005,2277,7,0.090994,0.000500,1.177659e-48,3.806244e-44,...,imk01643,622.228322,622.815466,348.406914,133.607561,True,fve,0.000778,1.907808e-47,3.853822e-43
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0006,2277,7,0.007872,0.006997,5.606826e-03,3.841969e-01,...,imk01220,128.009940,143.542635,117.889387,40.459149,False,fve,0.009290,7.569215e-03,3.939234e-01
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0007,2277,7,0.017145,0.000500,1.310140e-05,1.864352e-03,...,imk00895,117.075662,97.514651,58.983192,8.945764,False,fve,0.000778,2.526698e-05,2.849293e-03


,session_id,subject_id,dmd,synapse_id,stimulus_name,stimulus_label,n_positions,n_sequences,r0,rlast,...,overall_slope,overall_slope_norm,early_slope,late_slope,sequence_label,image_selectivity_score,ranking_score,image_rank_within_synapse,rank_basis,is_preferred_ranked_image
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,21,40,4.135909,-3.317766,...,0.176119,0.042583,-0.015838,0.653529,stable,-34.922383,45.880193,5,response_amplitude,False
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01220.tiff,imk01220,17,40,4.002009,3.400045,...,0.314888,0.078682,0.139810,1.405141,stable,64.178829,130.824089,2,response_amplitude,False
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk00459.tiff,imk00459,16,40,2.642996,11.732948,...,0.526522,0.199214,0.181800,1.475654,stable,-14.799073,63.128745,4,response_amplitude,False
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01643.tiff,imk01643,13,39,3.792312,5.586604,...,0.053110,0.014005,0.095214,-0.118465,stable,156.407801,209.877494,1,response_amplitude,True
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk00942.tiff,imk00942,14,39,9.852383,9.400597,...,-0.567097,-0.057559,-0.757631,0.229945,stable,-96.587349,-6.975492,7,response_amplitude,False


,session_id,subject_id,dmd,synapse_id,stimulus_name,stimulus_label,position_category,sequence_position,counts,n_sequences,...,epoch_label,binned_position_center,binned_response_amplitude,binned_response_amplitude_norm,binned_counts,image_selectivity_score,ranking_score,image_rank_within_synapse,rank_basis,is_preferred_ranked_image
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,repeated,0,40.0,40,...,early,2.888889,4.678374,1.13116,270.0,-34.922383,45.880193,5,response_amplitude,False
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,repeated,1,40.0,40,...,early,2.888889,4.678374,1.13116,270.0,-34.922383,45.880193,5,response_amplitude,False
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,repeated,2,40.0,40,...,early,2.888889,4.678374,1.13116,270.0,-34.922383,45.880193,5,response_amplitude,False
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,repeated,3,40.0,40,...,early,2.888889,4.678374,1.13116,270.0,-34.922383,45.880193,5,response_amplitude,False
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,repeated,4,40.0,40,...,early,2.888889,4.678374,1.13116,270.0,-34.922383,45.880193,5,response_amplitude,False


...

,session_id,subject_id,dmd,synapse_id,n_image_trials,n_images_tested,fve_image,p_shuffle_fve,p_kw,p_manova,...,preferred_image,preferred_mean,preferred_median,preferred_vs_rest_effect,preferred_vs_next_effect,is_tuned,tuning_method,q_shuffle_fve,q_kw,q_manova
0,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,2358,7,0.003055,0.312844,0.135597,0.001453,...,268048,59.674307,0.570121,-4.244304,6.624389,False,fve,0.312844,0.135597,0.001453


,session_id,subject_id,dmd,synapse_id,stimulus_name,stimulus_label,n_positions,n_sequences,r0,rlast,...,overall_slope,overall_slope_norm,early_slope,late_slope,sequence_label,image_selectivity_score,ranking_score,image_rank_within_synapse,rank_basis,is_preferred_ranked_image
0,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,stimuli\images_B\imk01333.tiff,imk01333,16,41,3.251047,9.101991,...,0.176233,0.054208,0.143708,0.295131,stable,23.959234,53.049918,2,response_amplitude,False
1,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,stimuli\images_B\100075.tiff,100075,12,41,5.846442,4.276362,...,0.062968,0.010770,0.128614,-0.065285,stable,4.706967,36.547975,4,response_amplitude,False
2,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,stimuli\images_B\69022.tiff,69022,14,41,4.139674,2.922333,...,-0.229843,-0.055522,-0.097099,-0.697211,stable,9.523214,40.676187,3,response_amplitude,False
3,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,stimuli\images_B\McGill_stairs.tiff,McGill_stairs,13,42,3.433658,5.620232,...,0.388975,0.113283,0.261582,0.773428,stable,-23.648170,12.243572,6,response_amplitude,False
4,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,stimuli\images_B\216066.tiff,216066,15,42,1.891201,-0.229533,...,0.089876,0.047523,-0.035940,0.485189,stable,-36.478776,1.245909,7,response_amplitude,False


,session_id,subject_id,dmd,synapse_id,stimulus_name,stimulus_label,position_category,sequence_position,counts,n_sequences,...,epoch_label,binned_position_center,binned_response_amplitude,binned_response_amplitude_norm,binned_counts,image_selectivity_score,ranking_score,image_rank_within_synapse,rank_basis,is_preferred_ranked_image
0,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,stimuli\images_B\imk01333.tiff,imk01333,repeated,0,40.0,41,...,early,2.5,3.023165,0.929905,240.0,23.959234,53.049918,2,response_amplitude,False
1,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,stimuli\images_B\imk01333.tiff,imk01333,repeated,1,40.0,41,...,early,2.5,3.023165,0.929905,240.0,23.959234,53.049918,2,response_amplitude,False
2,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,stimuli\images_B\imk01333.tiff,imk01333,repeated,2,40.0,41,...,early,2.5,3.023165,0.929905,240.0,23.959234,53.049918,2,response_amplitude,False
3,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,stimuli\images_B\imk01333.tiff,imk01333,repeated,3,40.0,41,...,early,2.5,3.023165,0.929905,240.0,23.959234,53.049918,2,response_amplitude,False
4,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,stimuli\images_B\imk01333.tiff,imk01333,repeated,4,40.0,41,...,early,2.5,3.023165,0.929905,240.0,23.959234,53.049918,2,response_amplitude,False


,session_id,subject_id,dmd,synapse_id,n_images_with_sequences,median_seq_slope,median_overall_slope,median_early_slope,median_late_slope,median_adaptation_index,median_r0,median_rlast,median_rterminal,median_terminal_minus_last,median_early_minus_late,seq_p,sequence_class,seq_q
0,838410_2026-03-20_10-00-59,838410,DMD2,DMD2_syn0012,7,0.153581,0.153581,0.128614,0.295131,-0.241507,3.433658,5.620232,2.560929,-3.059303,-2.146473,0.15625,stable,0.15625


\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-20_10-00-59\analysis\derived\glutamate\glutamate_analysis
['activation_event_table.csv', 'activation_event_table.parquet', 'activation_summary_table.csv', 'activation_summary_table.parquet', 'glutamate_analysis_metadata.json', 'sequence_per_image_table.csv', 'sequence_per_image_table.parquet', 'sequence_position_table.csv', 'sequence_position_table.parquet', 'sequence_summary_table.csv', 'sequence_summary_table.parquet', 'tuning_per_image_table.csv', 'tuning_per_image_table.parquet', 'tuning_summary_table.csv', 'tuning_summary_table.parquet']
